# 06 · Enhancement 2: Foveated DA3, mixed per-view resolutions in one pass
Training-free enhancement on the **released** weights (setting **[official]**); the network is not
changed. DA3 runs every view at one resolution, so the token count grows with views × resolution and
cross-view attention grows quadratically with it. Foveated DA3 gives a few **focus views** the full
resolution (504 px) and the other **context views** a quarter of the tokens (252 px), all in **one
forward pass**. Region level: a **virtual zoom camera** (exact rotation of the source camera) turns a
region of a 6048 × 4032 photo into a 504-px view that DA3 processes with the scene views, its camera
given through E_c. Plan and novelty: `docs/da3_enhancement_report.md` §4.3 · how to run and save:
`docs/ENHANCEMENTS_RUNBOOK.md` §3 · write-up: `reports/enhancement_2_foveated.md`.

Why it is possible: the released backbone gives every patch token the same constant RoPE position in
the cross-view blocks, so cross-view attention is well defined over views with different token
counts; patch embedding, position embedding, within-view blocks, Dual-DPT and the camera head work
per view and run once per resolution group (`src/da3/enhancements/foveated/routing.py`). With all
views at one resolution the routing is **bit-identical** to the original (part 0 checks it).

**Statistics are pre-registered** (`docs/FOVEATED_ANALYSIS_PLAN.md`, constants in
`configs/foveated_analysis.yaml`, whose SHA-256 goes into every result record): the scene is the unit
of analysis; every comparison is paired, with a 95% scene-bootstrap CI; every gate is **PASS / FAIL /
INCONCLUSIVE** read off its CI, with the margins at which the decision would change. Margins: the
pre-registered 3%, raised to the noise floor of a treatment-blind A/A run (the original DA3 with only
the reference view moved), capped at 6%. One primary configuration per gate; mitigations are
secondary (simultaneous CIs); any selection happens on a dev half of the scenes and is confirmed on
the held-out half.

| Part | What | Compared with | Data | Time (T4, both models) |
|---|---|---|---|---|
| 0 · routing check | equal resolutions → bit-identical outputs | `DA3.forward` | robot video frames | ~5 min |
| 1 · probe = gates 1 + 3 | 3 × 16 views / scene: 8 focus at 504 px + 8 context at 252 / 280 px (plain, attention-balanced, 3 random focus choices), focus views alone, A/A | all-504, uniform 378 / 336 / 280, all-252 | HiRoom, 7Scenes | ~1 h |
| 2 · large-view protocol | 100 … 1000 views in one pass, 10% focus views, repeated timings | all-504, uniform at equal tokens, fewer 504-px views at equal tokens, all-252 | 7Scenes, all 7 sequences | ~1.5 h (DA3-Small) |
| 3 · Sec. 6 protocol | released harness (pose AUC, TSDF F1), 25% / 10% focus views, paired CIs | uniform 504 (original DA3) / 336 / 252 | HiRoom, 7Scenes | 3–6 h (TSDF on CPU); ~1 h pose-only |
| 4 · region zoom = gate 2 | zoom views from the full-resolution photos through E_c; oracle, random and texture regions | global 504 / 896 px, naive crop, monocular | ETH3D, 11 scenes (15 GB): **own session** | ~1.5 h |

Settings: **Accelerator = GPU (T4 / P100)**, **Internet = on**. Choose the parts with `RUN`. A
session that stops early can be resumed: attach its output and list it in `RESUME_FROM`; finished
parts are skipped. Outputs in `/kaggle/working/foveated/`: JSON per part, figures, `summary.md`,
`gates.json`, and `results/` (standard result records: **copy them into the repo under
`experiments/results/` and commit**, so notebook 09 can compare all enhancements).

In [ ]:
# --- repo setup (identical in every notebook) ---------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess, sys
os.chdir("/kaggle/working")
url = REPO_URL
try:  # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
sys.path.insert(0, os.path.abspath("src"))   # the editable install is only seen by new processes, not this kernel
!git log -1 --format="commit %H  %cd"

In [ ]:
# --- official DA3 repo (reference / parity oracle / benchmark harness), pinned commit ----------
import os, subprocess
if not os.path.exists("/kaggle/working/Depth-Anything-3"):
    subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
                    "/kaggle/working/Depth-Anything-3"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/Depth-Anything-3", "checkout", "3d835ec1a5802d64a8b8b15f817a1ab54809bfe4"], check=True)
OFFICIAL_SRC = "/kaggle/working/Depth-Anything-3/src"
!pip install -q -r requirements-eval.txt
# Open3D self-test: does TSDF fusion (depth maps -> mesh) work in this environment?
!python scripts/check_open3d.py --out /kaggle/working/open3d_check.json

In [ ]:
import json
chk = json.load(open("/kaggle/working/open3d_check.json"))
print("Open3D TSDF OK:", chk["legacy_tsdf_ok"], "| open3d", chk.get("open3d"))

### Configuration

In [ ]:
MODELS = ["da3-small", "da3-base"]          # released weights (Apache-2.0)
DATASETS = ["hiroom", "7scenes"]            # view level: the datasets with per-frame GT depth
MAX_SCENES = 0                              # 0 = all scenes; e.g. 2 for a quick end-to-end check ("limited")
RUN = {"parity": True, "probe": True, "scaling": True, "sec6": True, "zoom": False}   # zoom: own session
RESUME_FROM = []                            # outputs of earlier sessions, e.g. ["/kaggle/input/<name>/foveated"]

PLAN = "configs/foveated_analysis.yaml"      # pre-registered analysis plan: do NOT edit after the runs

# 1 · probe (gates 1 and 3): half of the views are focus views at 504 px
PROBE_VIEWS = 16
PROBE_SUBSETS = 3                           # interleaved view subsets per scene (the scene stays the unit)
PROBE_UNIFORM = [504, 378, 336, 280, 252]
PROBE_FOVEATED = ["context_res=252", "context_res=280", "context_res=252,balance=1"]   # first = primary (plan)
PROBE_FOCUS_SEEDS = [1, 2, 3]               # robustness: random focus views for the primary configuration
# 2 · large-view protocol (7Scenes sequences: 1000 frames each, stairs 500)
SCALING_MODELS = ["da3-small"]
SCALING_SCENES = ["chess", "fire", "heads", "office", "pumpkin", "redkitchen", "stairs"]
SCALING_VIEWS = [100, 250, 500, 1000]
SCALING_FRACTION = 0.1
TIMING_REPEATS, TIMING_SCENES = 3, 2        # median of 3 timed passes on the first 2 scenes
# 3 · Sec. 6 protocol (released harness, <= 100 frames, seed 42)
SEC6_MODES = ["pose", "recon_unposed"]      # ["pose"] = fast run without TSDF fusion
SEC6_UNIFORM = [504, 336, 252]              # 504 = the original DA3, the reference row
SEC6_FOVEATED = ["fraction=0.25,context_res=252", "fraction=0.25,context_res=252,balance=1",
                 "fraction=0.1,context_res=252"]
KEEP_EXPORTS = False                        # ~2 GB per run; the per-view analysis runs before they are deleted
# 4 · region zoom (ETH3D zip is 15 GB; only the photos and the source views' GT maps are extracted)
ZOOM_SCENES = ["courtyard", "electro", "kicker", "pipes", "relief", "delivery_area", "facade", "office",
               "playground", "relief_2", "terrains"]
ZOOM_SELECTORS = ["oracle", "random", "texture"]   # oracle = primary (report); random / texture = deployable

OUT = "/kaggle/working/foveated"            # JSON per part, figures, summary.md, gates.json
RESULTS = f"{OUT}/results"                  # standard result records (commit these)
WORK = "/tmp/fov_work"                      # harness workspace and Sec. 6 exports (large, not kept)
BENCH = "/tmp/da3bench"
DS = " ".join(DATASETS)
LIM = f"_limited{MAX_SCENES}" if MAX_SCENES else ""
H = f"--official-src {OFFICIAL_SRC} --bench-root {BENCH} --work-root {WORK}"
def sec6_tag(m, cfg):
    return f"fov_{m}_{'-'.join(DATASETS)}{LIM}__{cfg}"
os.makedirs(RESULTS, exist_ok=True)
for src in RESUME_FROM:
    !cp -rn {src}/* {OUT}/ && ls {OUT}

### Data: DA3-BENCH subsets (HF `depth-anything/DA3-BENCH`)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
os.makedirs(BENCH, exist_ok=True)
need = set(DATASETS) if (RUN["probe"] or RUN["sec6"]) else set()
if RUN["scaling"]:
    need.add("7scenes")
for d in sorted(need):
    if not os.path.exists(f"{BENCH}/{d}"):
        z = hf_hub_download("depth-anything/DA3-BENCH", f"{d}.zip", repo_type="dataset", local_dir="/tmp/zips")
        zipfile.ZipFile(z).extractall(BENCH); os.remove(z)
!du -sh {BENCH}/* 2>/dev/null; df -h /tmp | tail -1

### 0 · Routing check: equal resolutions must reproduce the original DA3
Bit-identical outputs (depth, confidence, rays, camera head) in fp32 and fp16, with the released
reference-view selection and with poses through E_c; round-off only when same-size views are split
into two groups; permuting views at mixed resolution only permutes the outputs. The notebook stops
if a check fails.

In [ ]:
if RUN["parity"]:
    !python scripts/extract_frames.py /kaggle/working/Depth-Anything-3/assets/examples/robot_unitree.mp4 \
        /kaggle/working/samples/robot_unitree --frames 8 --start 0.1 --end 0.6
    for m in MODELS:
        !python scripts/foveated_eval.py parity --weights official:{m} --images /kaggle/working/samples/robot_unitree \
            --views 8 --out {OUT}/parity_{m}.json
for m in MODELS:
    p = f"{OUT}/parity_{m}.json"
    if os.path.exists(p):
        assert json.load(open(p))["all_pass"], f"routing check failed for {m}: see {p}"

### 1 · Probe: gate 1 (uniform-resolution curve) and gate 3 (cross-scale probe)
3 interleaved subsets of 16 evenly spaced views per scene; every other view is a focus view.
- **G1a headroom:** all-252 is worse than all-504 by more than the margin.
- **G1b context informative:** the focus views are better with the context views than alone.
- **G3 (primary 504/252):** focus-view AbsRel non-inferior to all-504 **and** context-pair AUC@30
  non-inferior to all-252.
- Secondary: 280-px context views, attention-mass balancing. Robustness: 3 random focus choices.
- The A/A run (all-504, reference view moved) sets the noise floor of the margins.

In [ ]:
if RUN["probe"]:
    unif = " ".join(map(str, PROBE_UNIFORM))
    fov = " ".join(f'"{s}"' for s in PROBE_FOVEATED)
    seeds = " ".join(map(str, PROBE_FOCUS_SEEDS))
    for m in MODELS:
        if not os.path.exists(f"{OUT}/probe_{m}.json"):
            !python scripts/foveated_eval.py probe {H} --weights official:{m} --datasets {DS} \
                --max-scenes {MAX_SCENES} --n-views {PROBE_VIEWS} --subsets {PROBE_SUBSETS} --uniform {unif} \
                --foveated {fov} --focus-seeds {seeds} --plan {PLAN} --out {OUT}/probe_{m}.json
for m in MODELS:
    p = f"{OUT}/probe_{m}.json"
    if os.path.exists(p):
        an = json.load(open(p))["analysis"]
        print(f"{m} · margins: depth {100 * an['margins']['depth']:.1f}%, pose {100 * an['margins']['pose']:.1f}%")
        for k, g in an["gates"].items():
            print(f"{m} · {g['result']:12s} [{g['role']}] {k}: {g['value']}")

### 2 · Large-view protocol: time, memory and accuracy from 100 to 1000 views per pass
Full 7Scenes sequences (GT poses for every frame). 10% focus views at 504 px, the rest at 252 px, vs
all at 504 px (until it runs out of memory), uniform resolution with the same token count, fewer
504-px views with the same token count (the focus views included), and all at 252 px. Accuracy is
read on the focus views, which every configuration contains.

In [ ]:
if RUN["scaling"]:
    scenes, views = " ".join(SCALING_SCENES), " ".join(map(str, SCALING_VIEWS))
    for m in SCALING_MODELS:
        if not os.path.exists(f"{OUT}/scaling_{m}.json"):
            !python scripts/foveated_eval.py scaling {H} --weights official:{m} --scenes {scenes} \
                --n-views {views} --fraction {SCALING_FRACTION} --balance --timing-repeats {TIMING_REPEATS} \
                --timing-scenes {TIMING_SCENES} --out {OUT}/scaling_{m}.json

### 3 · Sec. 6 protocol: released harness with uniform and foveated resolutions
Every run goes through `scripts/eval_bench.py` (`--process-res` / `--meter` for uniform resolutions,
`--foveated` for mixed ones) and writes a standard record; the reference row is the original DA3
(504 px, camera-head poses). Then the per-view analysis (focus vs context views, depth vs GT, pose
AUC on view subsets) reads the exports before they are deleted.

In [ ]:
from da3.enhancements.foveated import FoveationSpec
if RUN["sec6"]:
    if "recon_unposed" in SEC6_MODES:
        assert chk["legacy_tsdf_ok"], "Open3D TSDF fusion is broken here: use SEC6_MODES = ['pose'] or send open3d_check.json"
    runs = [(f"u{r}", f"--process-res {r} --meter", "baseline", "head" if r == 504 else f"head_res{r}") for r in SEC6_UNIFORM]
    runs += [(FoveationSpec.parse(s).name, f'--foveated "{s}"', "foveated", FoveationSpec.parse(s).name) for s in SEC6_FOVEATED]
    fractions = " ".join(sorted({f"{FoveationSpec.parse(s).fraction:g}" for s in SEC6_FOVEATED}))
    modes = " ".join(SEC6_MODES)
    for m in MODELS:
        ref = sec6_tag(m, "u504")
        for cfg, flags, enh, var in runs:
            t = sec6_tag(m, cfg)
            if os.path.exists(f"{OUT}/sec6views_{t}.json"):
                print("already done:", t)
                continue
            !python scripts/eval_bench.py {H} --weights official:{m} --datasets {DS} --modes {modes} \
                --max-scenes {MAX_SCENES} --tag {t} --enhancement {enh} --variant {var} --baseline-tag {ref} \
                --results-root {RESULTS} {flags}
            !python scripts/foveated_eval.py sec6views {H} --tag {t} --config {cfg} --datasets {DS} \
                --fractions {fractions} --out {OUT}/sec6views_{t}.json
            if not KEEP_EXPORTS:
                !rm -rf {WORK}/workspace/evaluation/{t}

### 4 · Region zoom (gate 2): virtual zoom cameras on ETH3D
Run it in its own session (`RUN = {"zoom": True}`, the other parts False; list the first session's
output in `RESUME_FROM` so the final analysis has every part). Per scene, 2 source views; per view one
region (1/4 of the image side, so the zoom view has 4× the global view's pixel density) from each
selector: **oracle** (most geometric detail in the GT: the report's feasibility gate, primary),
**random** and **texture** (image only): deployable, secondary. Gate 2: region AbsRel ≥ 10% better
than global 896 px and the zoom rotation R_v recovered within 1°, read off scene-bootstrap CIs.

In [ ]:
if RUN["zoom"]:
    from huggingface_hub import hf_hub_download
    zip_path = hf_hub_download("depth-anything/DA3-BENCH", "eth3d.zip", repo_type="dataset", local_dir="/tmp/zips")
    scenes, selectors = " ".join(ZOOM_SCENES), " ".join(ZOOM_SELECTORS)
    for m in MODELS:
        if not os.path.exists(f"{OUT}/zoom_{m}.json"):
            !python scripts/foveated_eval.py zoom {H} --weights official:{m} --eth3d-zip {zip_path} \
                --scenes {scenes} --selectors {selectors} --plan {PLAN} --fig-dir {OUT}/zoom_{m} \
                --out {OUT}/zoom_{m}.json

### 5 · Results: pre-registered gates, tables, figures, result records
The analysis re-derives every decision from the saved per-scene data and the plan, and warns if a part
ran under a different plan version. `gates_<model>.png`: one row per criterion, CI bar and estimate
coloured by the decision (written next to it), dashed tick = threshold.

In [ ]:
models = " ".join(sorted(set(MODELS) | set(SCALING_MODELS)))
!python scripts/foveated_eval.py analyse --out-dir {OUT} --results-root {RESULTS} --models {models} --plan {PLAN}

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f"{OUT}/summary.md").read()))

In [ ]:
for m in sorted(set(MODELS) | set(SCALING_MODELS)):
    for f in [f"gates_{m}.png", f"probe_{m}.png", f"sec6_{m}.png", f"scaling_{m}.png", f"gates_zoom_{m}.png",
              f"zoom_{m}/zoom_examples.png"]:
        p = f"{OUT}/{f}"
        if os.path.exists(p):
            print(m, "·", f); display(Image(p, width=1100))

### 6 · Save: download `foveated_results.zip`, copy `foveated/results/*` into the repo's `experiments/results/`, commit

In [ ]:
!cd /kaggle/working && rm -f foveated_results.zip && zip -qr foveated_results.zip foveated && ls -la foveated_results.zip
!find {RESULTS} -name "*.json" | sort